In [ ]:
import base64
import math
import numpy as np
from Crypto.Util.number import getPrime, inverse, bytes_to_long, long_to_bytes
from collections import defaultdict
from itertools import product
from functools import lru_cache

with open('./english_quadgrams.txt', 'r') as f:
    english_quadgrams = set(word.upper() for word in f.readlines())

with open('./english_words.txt', 'r') as f:
    english_words = set(word.upper().strip() for word in f.readlines())

def caesar_cipher(message, shift, isEncrypting = True):
    encoded_message = ''
    for char in message:
        if char.isalpha():
            offset = ord('A') if char.isupper() else ord('a')
            shifted_char = chr((ord(char) - offset + (shift if isEncrypting else -shift)) % 26 + offset)
            encoded_message += shifted_char
        else:
            encoded_message += char
    return encoded_message

def base64_encode(message, isEncrypting = True):
    message_bytes = message.encode('utf-8')
    try:
        encoded_bytes = base64.b64encode(message_bytes) if isEncrypting else base64.b64decode(message_bytes)
        return encoded_bytes.decode('utf-8')
    except Exception as e:
        print("Exception in base64_encode: ", e)
        return ''

def atbash_cipher(message):
    result = ''
    for char in message:
        if char.isalpha():
            base = ord('A') if char.isupper() else ord('a')
            transformed = chr(base + (25 - (ord(char) - base)))
            result += transformed
        else:
            result += char
    return result

def vigenere_cipher(message, key, isEncrypting = True):
    result = ''
    key = key.lower()
    key_index = 0
    for char in message:
        if char.isalpha():
            base = ord('A') if char.isupper() else ord('a')
            text_ord = ord(char) - base
            key_ord = ord(key[key_index % len(key)]) - ord('a')
            shifted = (text_ord + key_ord) % 26 if isEncrypting else (text_ord - key_ord + 26) % 26
            result += chr(base + shifted)
            key_index += 1
        else:
            result += char
    return result

def generate_playfair_matrix(keyword):
    matrix = []
    seen = set()
    keyword = keyword.upper().replace('J', 'I')

    for char in keyword:
        if char.isalpha() and char not in seen:
            matrix.append(char)
            seen.add(char)

    for char in "ABCDEFGHIKLMNOPQRSTUVWXYZ":  # J is excluded
        if char not in seen:
            matrix.append(char)
            seen.add(char)

    return [matrix[i:i+5] for i in range(0, 25, 5)]

def find_position(matrix, char):
    for row_idx, row in enumerate(matrix):
        if char in row:
            return row_idx, row.index(char)
    raise ValueError(f"Character {char} not found in matrix.")

def process_text(text):
    text = text.upper().replace('J', 'I')
    text = ''.join([c for c in text if c.isalpha()])
    pairs = []
    i = 0

    while i < len(text):
        a = text[i]
        if i + 1 < len(text):
            b = text[i + 1]
            if a == b:
                b = 'X'
                i += 1
            else:
                i += 2
        else:
            b = 'X'
            i += 1
        pairs.append((a, b))
    return pairs

def playfair_cipher(text, keyword, isEncrypted=True):
    matrix = generate_playfair_matrix(keyword)
    pairs = process_text(text)

    result = ''
    for a, b in pairs:
        row1, col1 = find_position(matrix, a)
        row2, col2 = find_position(matrix, b)

        if row1 == row2:
            if isEncrypted:
                result += matrix[row1][(col1 + 1) % 5]
                result += matrix[row2][(col2 + 1) % 5]
            else:
                result += matrix[row1][(col1 - 1) % 5]
                result += matrix[row2][(col2 - 1) % 5]
        elif col1 == col2:
            if isEncrypted:
                result += matrix[(row1 + 1) % 5][col1]
                result += matrix[(row2 + 1) % 5][col2]
            else:
                result += matrix[(row1 - 1) % 5][col1]
                result += matrix[(row2 - 1) % 5][col2]
        else:
            result += matrix[row1][col2]
            result += matrix[row2][col1]

    return result

def railfence_encrypt(message, num_rails):
    if num_rails <= 1:
        return message
    rails = ['' for _ in range(num_rails)]
    rail = 0
    direction = 1
    for char in message:
        rails[rail] += char
        rail += direction
        if rail == 0 or rail == num_rails - 1:
            direction *= -1
    return ''.join(rails)

def railfence_decrypt(message, num_rails):
    if num_rails <= 1:
        return message
    pattern = ['' for _ in range(len(message))]
    rail = 0
    direction = 1
    for i in range(len(message)):
        pattern[i] = rail
        rail += direction
        if rail == 0 or rail == num_rails - 1:
            direction *= -1
    rail_lengths = [pattern.count(r) for r in range(num_rails)]
    rails = []
    idx = 0
    for length in rail_lengths:
        rails.append(message[idx:idx+length])
        idx += length
    result = ''
    rail_indices = [0]*num_rails
    for r in pattern:
        result += rails[r][rail_indices[r]]
        rail_indices[r] += 1
    return result

def get_order(keyword):
    return sorted(range(len(keyword)), key=lambda k: keyword[k])

def columnar_encrypt(message, keyword):
    message = message.replace(" ","")
    num_cols = len(keyword)
    num_rows = math.ceil(len(message)/num_cols)
    padded_length = num_cols*num_rows
    message += 'X' * (padded_length - len(message))
    matrix = [message[i:i+num_cols] for i in range(0,len(message),num_cols)]
    order = get_order(keyword)
    ciphertext = ''
    for col in order:
        for row in matrix:
            ciphertext+= row[col]
    return ciphertext

def columnar_decrypt(message, keyword):
    num_cols = len(keyword)
    num_rows = math.ceil(len(message)/num_cols)
    order = get_order(keyword)
    col_lengths = [num_rows]*num_cols
    total_cells = num_cols*num_rows
    extra = total_cells-len(message)
    for i in reversed(order):
        if extra <= 0:
            break
        col_lengths[i] -= 1
        extra -= 1
    cols = {}
    idx = 0
    for col_index in order:
        length = col_lengths[col_index]
        cols[col_index] = message[idx:idx+length]
        idx += length
    text = ''
    for row in range(num_rows):
        for col in range(num_cols):
            if row < len(cols[col]):
                text += cols[col][row]
    return text

def scytale_script(message, circumference, isEncrypted=True):
    message = message.replace(" ","")
    while len(message) % circumference != 0:
        message += "_"
    rows = len(message) // circumference
    result = '' if isEncrypted else ['']*len(message)
    index = 0
    for i in range(circumference):
        for j in range(rows):
            if isEncrypted:
                result += message[j * circumference + i]
            else:
                result[j * circumference + i] = message[index]
                index += 1
    return result if isEncrypted else ''.join(result).rstrip("_")

def mod_inverse(a,m):
    a %= m
    for x in range(1,m):
        if (a*x) % m == 1:
            return x
    raise ValueError(f"No modular inverse for a = {a} under modulo {m}")

def affine_encrypt(text, a, b, isEncrypted=True):
    if math.gcd(a,26) != 1:
        raise ValueError("Key 'a' must be coprime with 26.")
    a_inv = mod_inverse(a,26)
    result = ''
    for char in text.upper():
        if char.isalpha():
            x = ord(char) - ord('A')
            encrypted_char = (a * x + b) % 26 if isEncrypted else (a_inv * (x - b)) % 26
            result += chr(encrypted_char + ord('A'))
        else:
            result += char
    return result

def matrix_mod_inverse(matrix, modulus):
    """Find the inverse of a 2x2 matrix under modulo arithmetic."""
    det = int(np.round(np.linalg.det(matrix)))  # Determinant
    det_inv = mod_inverse(det % modulus, modulus)

    # Matrix of minors, then cofactor, then adjugate (for 2x2)
    if matrix.shape != (2, 2):
        raise ValueError("Only 2x2 matrices are supported in this example.")

    inv_matrix = np.array([[matrix[1][1], -matrix[0][1]],
                           [-matrix[1][0], matrix[0][0]]])
    inv_matrix = (det_inv * inv_matrix) % modulus
    return inv_matrix

def text_to_numbers(text):
    """Convert letters to numbers (A=0, B=1, ..., Z=25)."""
    return [ord(c) - ord('A') for c in text.upper() if c.isalpha()]

def numbers_to_text(numbers):
    """Convert numbers back to uppercase letters."""
    return ''.join(chr(n % 26 + ord('A')) for n in numbers)

def hill_encrypt(plaintext, key_matrix):
    """Encrypt plaintext using the Hill cipher and a given key matrix."""
    nums = text_to_numbers(plaintext)
    while len(nums) % key_matrix.shape[0] != 0:
        nums.append(0)  # Pad with 'A'

    ciphertext = []
    for i in range(0, len(nums), key_matrix.shape[0]):
        chunk = np.array(nums[i:i + key_matrix.shape[0]])
        enc = np.dot(key_matrix, chunk) % 26
        ciphertext.extend(enc)
    return numbers_to_text(ciphertext)

def hill_decrypt(ciphertext, key_matrix):
    """Decrypt ciphertext using the Hill cipher and the given key matrix."""
    nums = text_to_numbers(ciphertext)
    key_inv = matrix_mod_inverse(key_matrix, 26)

    plaintext = []
    for i in range(0, len(nums), key_matrix.shape[0]):
        try:
            chunk = np.array(nums[i:i + key_matrix.shape[0]])
            dec = np.dot(key_inv, chunk) % 26
            plaintext.extend(dec)
        except Exception as e:
            print("Hill exception: ",e)
            return ''
    return numbers_to_text(plaintext)

def generate_rsa_keys(bits=512):
    e = 65537
    while True:
        p = getPrime(bits)
        q = getPrime(bits)
        if p == q:
            continue
        n = p * q
        phi_n = (p-1)*(q-1)
        if inverse(e, phi_n):
            d = inverse(e, phi_n)
            break
    return {'public': (e,n), 'private': (d,n)}

def rsa_encrypt(plaintext: str, public_key):
    e, n = public_key
    m = bytes_to_long(plaintext.encode('utf-8'))
    if m >= n:
        raise ValueError("Message too long for the key size.")
    c = pow(m,e,n)
    return c

def rsa_decrypt(ciphertext: int, private_key):
    d,n = private_key
    m = pow(ciphertext,d,n)
    return long_to_bytes(m).decode('utf-8')

pigpen_cipher = {
    'A': '⍁', 'B': '⍂', 'C': '⍃', 'D': '⍄', 'E': '⍅', 'F': '⍆',
    'G': '⍇', 'H': '⍈', 'I': '⍉', 'J': '⍊', 'K': '⍋', 'L': '⍌',
    'M': '⍍', 'N': '⍎', 'O': '⍏', 'P': '⍐', 'Q': '⍑', 'R': '⍒',
    'S': '⍓', 'T': '⍔', 'U': '⍕', 'V': '⍖', 'W': '⍗', 'X': '⍘',
    'Y': '⍙', 'Z': '⍚'
}
reverse_pigpen = {v:k for k,v in pigpen_cipher.items()}

def encrypt_pigpen(message, isEncrypted=True):
    message = message.upper() if isEncrypted else message
    encrypted = ''
    for char in message:
        if char in pigpen_cipher:
            encrypted += pigpen_cipher[char]
        elif char in reverse_pigpen:
            encrypted += reverse_pigpen[char]
        else:
            encrypted += char
    return encrypted

bacon_cipher = {
    chr(i+ord('A')): format(i, '05b').replace('0','A').replace('1','B') for i in range(26)
}
reverse_bacon = {v:k for k,v in bacon_cipher.items()}

def encrypt_bacon(message, isEncrypted=True):
    encrypted = ''
    if isEncrypted:
        for char in message.upper():
            if char in bacon_cipher:
                encrypted += bacon_cipher[char]
    else:
        chunks = [message[i:i+5] for i in range(0,len(message),5)]
        for chunk in chunks:
            letter = reverse_bacon.get(chunk, '?')
            encrypted += letter
    return encrypted

def hide_bacon_in_cover(message, cover_text):
    bacon = encrypt_bacon(message)
    result = ''
    i = 0
    for char in cover_text:
        if char.isalpha() and i < len(bacon):
            result += char.upper() if bacon[i] == 'A' else char.lower()
            i += 1
        else:
            result += char
    return result

def reveal_bacon_from_cover(cover_text):
    pattern = ''
    for char in cover_text:
        if char.isalpha():
            pattern += 'A' if char.isupper() else 'B'
    return encrypt_bacon(pattern, False)

morse_cipher = {
    'A': '.-',     'B': '-...',   'C': '-.-.', 
    'D': '-..',    'E': '.',      'F': '..-.',
    'G': '--.',    'H': '....',   'I': '..',
    'J': '.---',   'K': '-.-',    'L': '.-..',
    'M': '--',     'N': '-.',     'O': '---',
    'P': '.--.',   'Q': '--.-',   'R': '.-.',
    'S': '...',    'T': '-',      'U': '..-',
    'V': '...-',   'W': '.--',    'X': '-..-',
    'Y': '-.--',   'Z': '--..',
    '0': '-----',  '1': '.----',  '2': '..---',
    '3': '...--',  '4': '....-',  '5': '.....',
    '6': '-....',  '7': '--...',  '8': '---..',
    '9': '----.',
    '&': '.-...',  "'": '.----.', '@': '.--.-.',
    ')': '-.--.-', '(': '-.--.',  ':': '---...',
    ',': '--..--', '=': '-...-',  '!': '-.-.--',
    '.': '.-.-.-', '-': '-....-', '+': '.-.-.',
    '"': '.-..-.', '?': '..--..', '/': '-..-.',
    ' ': '/'  # Space between words
}
reverse_morse = {v:k for k,v in morse_cipher.items()}

def encrypt_morse(message, isEncrypted=True):
    message = message.upper()
    encrypted = []
    if isEncrypted:
        for char in message:
            if char in morse_cipher:
                encrypted.append(morse_cipher[char])
            else:
                encrypted.append('?')
    else:
        words = message.split(' / ')
        for word in words:
            letters = word.split()
            encrypted.append(''.join(reverse_morse.get(letter, '?') for letter in letters))
    return ' '.join(encrypted)

@lru_cache(maxsize=10000)
def split_words(text):
    text = text.upper()
    if not text:
        return []
    best_split = None
    for i in range(3,len(text)+1):
        word = text[:i]
        if word in english_words and len(word) > 1:
            rest = split_words(text[i:])
            if rest is not None:
                candidate = [word] + rest
                if best_split is None or len(candidate) > len(best_split):
                    best_split = candidate
    if text in english_words:
        if best_split is None or len([text]) < len(best_split):
            best_split = [text]
    return best_split

def word_confidence(text: str):
    if not text:
        return 0
    words_split = split_words(text.lower().replace(' ', ''))
    if not words_split:
        return 0
    return sum(len(word)**2 for word in words_split)

vigenere_key_list = [word.upper() for word in english_words if len(word) <= 6]

def bruteforce(ciphertext, mode):
    best = {'key': None, 'result': '', 'score': 0}
    match mode:
        case 'Atbash':
            pt = atbash_cipher(ciphertext)
            s = word_confidence(pt)
            if s > best['score']:
                best.update({'key': None, 'result': pt, 'score': s})
        case 'Caesar':
            for shift in range(1,26):
                pt = caesar_cipher(ciphertext, shift, False)
                s = word_confidence(pt)
                if s > best['score']:
                    best.update({'key': shift, 'result': pt, 'score': s})
        case 'Affine':
            for a in range(1,26):
                if math.gcd(a,26) != 1:
                    continue
                for b in range(26):
                    pt = affine_encrypt(ciphertext, a, b, False)
                    s = word_confidence(pt)
                    if s > best['score']:
                        best.update({'key': (a,b), 'result': pt, 'score': s})
        case 'Vigenere':
            for k in vigenere_key_list:
                pt = vigenere_cipher(ciphertext, k, False)
                s = word_confidence(pt)
                if s > best['score']:
                    best.update({'key': k, 'result': pt, 'score': s})
        case 'Railfence':
            for r in range(2,min(len(ciphertext),10)):
                pt = railfence_decrypt(ciphertext, r)
                s = word_confidence(pt)
                if s > best['score']:
                    best.update({'key': r, 'result': pt, 'score': s})
        case 'Hill':
            letters = list(range(14))
            for a,b,c,d in product(letters, repeat=4):
                if math.gcd(a*d-b*c,26) != 1:
                    continue
                matrix = np.array([[a,b],[c,d]])
                pt = hill_decrypt(ciphertext, matrix)
                s = word_confidence(pt)
                if s > best['score']:
                    best.update({'key': matrix, 'result': pt, 'score': s})
        case 'Scytale':
            for dia in range(2, min(15, len(ciphertext))):
                pt = scytale_script(ciphertext, dia, False)
                s = word_confidence(pt)
                if s > best['score']:
                    best.update({'key': dia, 'result': pt, 'score': s})
        case 'Morse':
            pt = encrypt_morse(ciphertext, False)
            s = word_confidence(pt)
            if s > best['score']:
                best.update({'key': None, 'result': pt, 'score': s})
        case 'Pigpen':
            pt = encrypt_pigpen(ciphertext, False)
            s = word_confidence(pt)
            if s > best['score']:
                best.update({'key': None, 'result': pt, 'score': s})
        case 'Bacon':
            pt = encrypt_bacon(ciphertext, False)
            s = word_confidence(pt)
            if s > best['score']:
                best.update({'key': None, 'result': pt, 'score': s})
    return best

def format_meta(method, data):
    match method:
        case 'Caesar': return f"with shift {data.get('key', '?')}"
        case 'Affine': return f"with keys {data.get('key', '?')}"
        case 'Vigenere': return f"with key {data.get('key', '?')}"
        case 'Hill': return f"with matrix size {data.get('key', '?')}"
        case 'Railfence': return f"with {data.get('key', '?')} rails"
        case 'Scytale': return f"with diameter {data.get('key', '?')}"
        case _: return ''

def analyze_ciphertext(ciphertext):
    results = {}

    # Substitution ciphers
    caesar = bruteforce(ciphertext, 'Caesar')
    atbash = bruteforce(ciphertext, 'Atbash')
    vigenere = bruteforce(ciphertext, 'Vigenere')
    affine = bruteforce(ciphertext, 'Affine')
    hill = bruteforce(ciphertext, 'Hill')
    morse = bruteforce(ciphertext, 'Morse')
    pigpen = bruteforce(ciphertext, 'Pigpen')
    bacon = bruteforce(ciphertext, 'Bacon')

    # Transposition ciphers
    railfence = bruteforce(ciphertext, 'Railfence')
    scytale = bruteforce(ciphertext, 'Scytale')
   
    # Encodings
    base64 = base64_encode(ciphertext, False)

    # Build the dictionary
    results['Caesar'] = (caesar['result'], format_meta('Caesar', caesar))
    results['Atbash'] = (atbash['result'], '')
    results['Vigenere'] = (vigenere['result'], format_meta('Vigenere', vigenere))
    results['Affine'] = (affine['result'], format_meta('Affine', affine))
    results['Hill'] = (hill['result'], format_meta('Hill', hill))
    results['Railfence'] = (railfence['result'], format_meta('Railfence', railfence))
    results['Scytale'] = (scytale['result'], format_meta('Scytale', scytale))
    results['Base64'] = (base64, '')
    results['Morse'] = (morse['result'], '')
    results['Pigpen'] = (pigpen['result'], '')
    results['Bacon'] = (bacon['result'], '')

    scores = {name: word_confidence(output) for name, (output, shift) in results.items()}
    print(f"Input Ciphertext: {ciphertext}\n")
    for name, (output, shift) in results.items():
        print(f"{name} output: {output}")
        if shift != '' and scores[name] > 0: print(f"{shift}")
        print(f"Confidence: {scores[name]:.2f}\n")
    best_guess = max(scores, key=scores.get)
    print(f"Likely Cipher: {best_guess} {shift} (Confidence: {scores[best_guess]:.2f})")

message = input("Enter your message: ")
analyze_ciphertext(message)

Exception in base64_encode:  Incorrect padding
Caesar  with shift None
Atbash  
Vigenere  with key None
Affine  with keys None
Hill  with matrix size None
Railfence  with None rails
Scytale  with diameter None
Base64  
Morse  
Pigpen  
Bacon HELLOWORLD 
Input Ciphertext: AABBBAABAAABABBABABBABBBABABBAABBBABAAABABABBAAABB

Caesar output: 
Confidence: 0.00

Atbash output: 
Confidence: 0.00

Vigenere output: 
Confidence: 0.00

Affine output: 
Confidence: 0.00

Hill output: 
Confidence: 0.00

Railfence output: 
Confidence: 0.00

Scytale output: 
Confidence: 0.00

Base64 output: 
Confidence: 0.00

Morse output: 
Confidence: 0.00

Pigpen output: 
Confidence: 0.00

Bacon output: HELLOWORLD
Confidence: 50.00

Likely Cipher: Bacon  (Confidence: 50.00)
.... . .-.. .-.. --- / .-- --- .-. .-.. -..
⍈⍅⍌⍌⍏ ⍗⍏⍒⍌⍄
AABBBAABAAABABBABABBABBBABABBAABBBABAAABABABBAAABB


In [ ]:
{
    "index": 1,
    "timestamp": "2025-05-15 16:30:00",
    "votes": [{"voter_id": "abc123", "candidate": "Alice"}],
    "proof": 100,
    "previous_hash": "0"
}